# 17 — Combiner les ingrédients distincts de 08, 09, 12

- **08** : CatBoost 600 itér, features + te_origin(sm30) — la base (LB 0.3569).
- **09** apporte le **seed-bagging** (absent de 08/12).
- **12** apporte le **MLP** (absent de 08/09).

On combine tout : **CatBoost baggé (3 graines, 600 itér) ⊕ MLP baggé (3 graines)**, blend rank.
On compare à 08 sur la CV (recent2 = proxy privé). Boussole : LB ≈ last − 0.004.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6; WINDOWS = (5, 10, 20); SMOOTHING = 30
CAT_SEEDS = [42, 1, 7]; MLP_SEEDS = [42, 1, 7]
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X
def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)
def feats_train(df, ref):
    X = base_build(df, ref); X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING); return X
def feats_apply(df, ref):
    X = base_build(df, ref); mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm); return X
def clean(X): return X.replace([np.inf, -np.inf], np.nan)
def cat(seed):
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=seed, verbose=False)
def mlp(seed):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         FunctionTransformer(lambda x: np.clip(x, -5, 5)),
                         MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, batch_size=4096,
                                       learning_rate_init=1e-3, max_iter=60, early_stopping=True,
                                       n_iter_no_change=6, random_state=seed))
def rk(x): return np.argsort(np.argsort(x)) / (len(x) - 1)

## CV : CatBoost baggé (09) vs MLP baggé (12) vs blend des deux

In [ ]:
oof_cat = np.zeros(len(train)); oof_mlp = np.zeros(len(train))
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]; ref = train.iloc[tr_op]
    Xtr = feats_train(train.iloc[tr_op], ref); Xva = feats_apply(train.iloc[va_op], ref); yt = y_all[tr_op]
    oof_cat[va_op] = np.mean([cat(s).fit(Xtr, yt).predict_proba(Xva)[:, 1] for s in CAT_SEEDS], axis=0)
    Xtr_c, Xva_c = clean(Xtr), clean(Xva)
    oof_mlp[va_op] = np.mean([mlp(s).fit(Xtr_c, yt).predict_proba(Xva_c)[:, 1] for s in MLP_SEEDS], axis=0)
    print("fold ok")

rows = []
for _, va in folds_full:
    vo = va[op03[va]]
    rows.append((evaluate_ap(y_all[vo], oof_cat[vo]),
                 evaluate_ap(y_all[vo], oof_mlp[vo]),
                 evaluate_ap(y_all[vo], 0.5*rk(oof_cat[vo]) + 0.5*rk(oof_mlp[vo]))))
R = np.array(rows)
print("            catbag  mlpbag  blend")
for k, r in enumerate(R): print(f"fold {k}:   {r[0]:.4f}  {r[1]:.4f}  {r[2]:.4f}")
print(f"\nlast    :  {R[-1,0]:.4f}  {R[-1,1]:.4f}  {R[-1,2]:.4f}")
print(f"recent2 :  {R[-2:,0].mean():.4f}  {R[-2:,1].mean():.4f}  {R[-2:,2].mean():.4f}")
print("\nRéférence 08 (CatBoost seul) : last 0.3607 | recent2 0.3662")
print("-> le blend bat-il 0.3662 en recent2 ?")

## Soumission (si recent2 du blend > 0.3662)

In [ ]:
ref_full = train.iloc[np.where(op03)[0]]; yf = y_all[op03]
Xf = feats_train(ref_full, ref_full)
te_op = op03_mask(test).to_numpy(); test_op = test.iloc[np.where(te_op)[0]]
Xte = feats_apply(test_op, ref_full)
pc = np.mean([cat(s).fit(Xf, yf).predict_proba(Xte)[:, 1] for s in CAT_SEEDS], axis=0)
Xf_c, Xte_c = clean(Xf), clean(Xte)
pm = np.mean([mlp(s).fit(Xf_c, yf).predict_proba(Xte_c)[:, 1] for s in MLP_SEEDS], axis=0)
proba = 0.5 * rk(pc) + 0.5 * rk(pm)
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "17_catbag_mlpbag")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))